# Project 6 — Module 7: Aprendizaje de Máquina No Supervisado
## Lección 3: Data Preparation — Encoding, Outliers & Normalization

| | |
|---|---|
| **Author** | Jose Marcel Lopez Pino |
| **Framework** | CRISP-DM + LEAN |
| **Phase** | 3 — Data Preparation |
| **Module** | 7 — Aprendizaje de Máquina No Supervisado (Alkemy Bootcamp) |
| **Dataset** | Customer Segmentation — Kaggle (kaushiksuresh147) |
| **Date** | 2026-03 |

---

> **Executive Summary:**
> This notebook transforms raw customer data into a model-ready feature matrix.
> Steps: split train/test (80/20), drop non-informative columns, impute missing values,
> encode categoricals, remove outliers via IQR, and standardize with StandardScaler.
> The scaler is serialized for deployment. Outputs feed directly into notebook 04 (Modeling).

## Table of Contents

1. [CRISP-DM Phase 3 — Data Preparation](#1-crisp-dm-phase-3--data-preparation)
2. [Environment Setup](#2-environment-setup)
3. [Load Raw Data](#3-load-raw-data)
4. [Train / Test Split (80/20)](#4-train--test-split-8020)
5. [Drop & Separate Columns](#5-drop--separate-columns)
6. [Impute Missing Values](#6-impute-missing-values)
7. [Encode Categorical Variables](#7-encode-categorical-variables)
8. [Remove Outliers (IQR)](#8-remove-outliers-iqr)
9. [Normalize Features (StandardScaler)](#9-normalize-features-standardscaler)
10. [Save Processed Datasets](#10-save-processed-datasets)
11. [LEAN Filter — Waste Elimination Review](#11-lean-filter--waste-elimination-review)
12. [Decisions Log — Lesson 3](#12-decisions-log--lesson-3)
13. [Next Steps — Lesson 4 Preview](#13-next-steps--lesson-4-preview)

## 1. CRISP-DM Phase 3 — Data Preparation

**Goal:** Produce a clean, encoded, normalized feature matrix ready for distance-based
clustering algorithms. Every transformation is justified by model requirements —
no step is added without a direct impact on clustering quality (LEAN principle).

## 2. Environment Setup

In [1]:
# ===== Environment Setup =====
import warnings
warnings.formatwarning = lambda msg, *args, **kwargs: f'Warning: {msg}\n'
warnings.simplefilter('always')

# Python utilities
from pathlib import Path

# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning — preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Model persistence
import joblib

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Blues_d')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_RAW        = Path('../data/raw')
DATA_PROCESSED  = Path('../data/processed')
DATA_FINAL      = Path('../data/final')
REPORTS_FIGURES = Path('../reports/figures')
REPORTS_FIGURES.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

print('Environment ready.')
print(f'Data path   : {DATA_RAW}')
print(f'Figures path: {REPORTS_FIGURES}')

Environment ready.
Data path   : ..\data\raw
Figures path: ..\reports\figures


## 3. Load Raw Data

In [2]:
# Load raw dataset — no transformations at this stage
df = pd.read_csv(DATA_RAW / 'Train.csv')
print(f'Raw shape: {df.shape}')
df.head(3)

Raw shape: (8068, 11)


,ID,Gender,Ever_Married,Age,Graduated,Profession,Work_Experience,Spending_Score,Family_Size,Var_1,Segmentation
0,462809,Male,No,22,No,Healthcare,1.0,Low,4.0,Cat_4,D
1,462643,Female,Yes,38,Yes,Engineer,NaN,Average,3.0,Cat_4,A
2,466315,Female,Yes,67,Yes,Engineer,1.0,Low,1.0,Cat_6,B


## 4. Train / Test Split (80/20)

We generate our own test set from `Train.csv` via a stratified 80/20 split.
The Kaggle `Test.csv` is discarded — it lacks the `Segmentation` column
and is not reproducible under version control.

The test set is saved to `data/raw/test.csv` and used in notebook 06 (Deployment)
to verify that the serialized pipeline generalizes to unseen customers.

In [3]:
# Split before any transformation — prevents data leakage
df_train, df_test = train_test_split(
    df,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=df['Segmentation'] if 'Segmentation' in df.columns else None
)

# Save test set to raw — it is treated as unseen data from this point
df_test.to_csv(DATA_RAW / 'test.csv', index=False)

print(f'Train set: {df_train.shape[0]:,} rows ({df_train.shape[0]/len(df):.0%})')
print(f'Test set : {df_test.shape[0]:,} rows ({df_test.shape[0]/len(df):.0%})')
print(f'Test set saved to: {DATA_RAW}/test.csv')

Train set: 6,454 rows (80%)
Test set : 1,614 rows (20%)
Test set saved to: ..\data\raw/test.csv


## 5. Drop & Separate Columns

In [10]:
# Hold out target label — do NOT use during unsupervised modeling
# Kept separately for post-hoc external validation only (notebook 05)
y_true = df_train['Segmentation'].copy() if 'Segmentation' in df_train.columns else None

# Drop non-informative columns before modeling
drop_cols = ['ID', 'Segmentation']
df_model = df_train.drop(columns=[c for c in drop_cols if c in df_train.columns])

print(f'Model features shape: {df_model.shape}')
print(f'Columns: {df_model.columns.tolist()}')

Model features shape: (6454, 9)
Columns: ['Gender', 'Ever_Married', 'Age', 'Graduated', 'Profession', 'Work_Experience', 'Spending_Score', 'Family_Size', 'Var_1']


## 6. Impute Missing Values

In [11]:
# Numeric: median imputation — robust to outliers already present in raw data
numeric_cols = df_model.select_dtypes(include='number').columns.tolist()
for col in numeric_cols:
    median_val = df_model[col].median()
    df_model[col] = df_model[col].fillna(median_val)

# Categorical: mode imputation — preserves modal distribution
cat_cols = df_model.select_dtypes(include='object').columns.tolist()
for col in cat_cols:
    mode_val = df_model[col].mode()[0]
    df_model[col] = df_model[col].fillna(mode_val)

print(f'Missing after imputation: {df_model.isnull().sum().sum()}')

Missing after imputation: 0


See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.


## 7. Encode Categorical Variables

Encoding strategy (from `docs/data_dictionary.md`):
- **Binary** (Gender, Ever_Married, Graduated): label encoding 0/1
- **Ordinal** (Spending_Score): Low=0, Average=1, High=2
- **Nominal** (Profession, Var_1): one-hot encoding — no target variable available in unsupervised context

In [12]:
# Binary encoding
binary_map = {
    'Gender':       {'Male': 1, 'Female': 0},
    'Ever_Married': {'Yes': 1, 'No': 0},
    'Graduated':    {'Yes': 1, 'No': 0},
}
for col, mapping in binary_map.items():
    if col in df_model.columns:
        df_model[col] = df_model[col].map(mapping)

# Ordinal encoding — preserves natural order of spending behavior
if 'Spending_Score' in df_model.columns:
    df_model['Spending_Score'] = df_model['Spending_Score'].map(
        {'Low': 0, 'Average': 1, 'High': 2}
    )

# One-hot encoding for nominal categoricals
nominal_cols = [c for c in ['Profession', 'Var_1'] if c in df_model.columns]
df_model = pd.get_dummies(df_model, columns=nominal_cols, drop_first=False)

print(f'Shape after encoding: {df_model.shape}')
print(f'Columns: {df_model.columns.tolist()}')

Shape after encoding: (6454, 23)
Columns: ['Gender', 'Ever_Married', 'Age', 'Graduated', 'Work_Experience', 'Spending_Score', 'Family_Size', 'Profession_Artist', 'Profession_Doctor', 'Profession_Engineer', 'Profession_Entertainment', 'Profession_Executive', 'Profession_Healthcare', 'Profession_Homemaker', 'Profession_Lawyer', 'Profession_Marketing', 'Var_1_Cat_1', 'Var_1_Cat_2', 'Var_1_Cat_3', 'Var_1_Cat_4', 'Var_1_Cat_5', 'Var_1_Cat_6', 'Var_1_Cat_7']


## 8. Remove Outliers (IQR)

In [13]:
# Apply IQR only to continuous numeric columns — not to binary/ordinal encoded columns
continuous_cols = [c for c in ['Age', 'Work_Experience', 'Family_Size'] if c in df_model.columns]

n_before = len(df_model)

# Build mask: row survives only if it passes ALL column filters
mask = pd.Series(True, index=df_model.index)
for col in continuous_cols:
    q1 = df_model[col].quantile(0.25)
    q3 = df_model[col].quantile(0.75)
    iqr = q3 - q1
    mask &= df_model[col].between(q1 - 1.5 * iqr, q3 + 1.5 * iqr)

df_clean = df_model[mask].reset_index(drop=True)
n_after  = len(df_clean)

print(f'Rows before outlier removal: {n_before:,}')
print(f'Rows after outlier removal : {n_after:,}')
print(f'Removed: {n_before - n_after:,} ({(n_before - n_after) / n_before * 100:.1f}%)')

Rows before outlier removal: 6,454
Rows after outlier removal : 6,147
Removed: 307 (4.8%)


## 9. Normalize Features (StandardScaler)

StandardScaler transforms each feature to zero mean and unit variance.
Mandatory for KMeans and DBSCAN — both are distance-based algorithms
where unscaled features with larger numeric ranges dominate unfairly.

In [14]:
feature_cols = df_clean.columns.tolist()

# Fit scaler on training data only — never fit on test set
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(df_clean[feature_cols])

# Save scaler for deployment — reused in notebook 06 on test set
joblib.dump(scaler, DATA_PROCESSED / 'scaler.pkl')

print(f'Feature matrix shape: {X_scaled.shape}')
print(f'Mean (should be ~0) : {X_scaled.mean(axis=0).round(3)[:5]}')
print(f'Std  (should be ~1) : {X_scaled.std(axis=0).round(3)[:5]}')
print(f'Scaler saved        : {DATA_PROCESSED}/scaler.pkl')

Feature matrix shape: (6147, 23)
Mean (should be ~0) : [ 0. -0.  0. -0.  0.]
Std  (should be ~1) : [1. 1. 1. 1. 1.]
Scaler saved        : ..\data\processed/scaler.pkl


## 10. Save Processed Datasets

In [15]:
# Save clean unscaled dataset — used for cluster profiling in notebook 05
df_clean.to_csv(DATA_PROCESSED / 'customers_clean.csv', index=False)

# Save scaled feature matrix — used directly by clustering algorithms in notebook 04
np.save(DATA_PROCESSED / 'X_scaled.npy', X_scaled)

# Save feature column names — needed to reconstruct profiles in notebook 05
pd.Series(feature_cols).to_csv(DATA_PROCESSED / 'feature_cols.csv', index=False)

print(f'Saved: customers_clean.csv')
print(f'Saved: X_scaled.npy — shape {X_scaled.shape}')
print(f'Saved: feature_cols.csv — {len(feature_cols)} features')
print(f'Path : {DATA_PROCESSED}')

Saved: customers_clean.csv
Saved: X_scaled.npy — shape (6147, 23)
Saved: feature_cols.csv — 23 features
Path : ..\data\processed


## 11. LEAN Filter — Waste Elimination Review

| LEAN Question | Answer | Action |
|---------------|--------|--------|
| Does every step link to model quality? | ✅ Each step prevents distance metric distortion | Proceed |
| Is the train/test split necessary? | ✅ Required for deployment validation in notebook 06 | Proceed |
| Is IQR sufficient? | ✅ Conservative removal appropriate for this dataset size | Proceed |
| Is StandardScaler justified? | ✅ KMeans and DBSCAN are distance-based — scaling is mandatory | Proceed |
| Is saving feature_cols.csv necessary? | ✅ Needed to reconstruct cluster profiles without re-running notebook 03 | Proceed |

## 12. Decisions Log — Lesson 3

| # | Decision | Rationale | Alternatives Considered | LEAN Value? |
|---|----------|-----------|------------------------|-------------|
| 1 | Split 80/20 before any transformation | Prevents data leakage — scaler fitted on train only | Split after preprocessing | ✅ |
| 2 | Stratify split on Segmentation | Preserves class balance in both sets | Random split | ✅ |
| 3 | Median imputation for numerics | Robust to outliers already present in raw data | Mean imputation | ✅ |
| 4 | Mode imputation for categoricals | Preserves modal distribution | Drop rows with missing values | ✅ |
| 5 | IQR factor=1.5 on continuous cols only | Conservative removal; binary/ordinal cols excluded | Z-score ±3 on all cols | ✅ |
| 6 | StandardScaler over MinMaxScaler | Distance-based algorithms require unit variance; MinMaxScaler sensitive to residual outliers | MinMaxScaler | ✅ |
| 7 | One-hot for Profession/Var_1 | No target variable available in unsupervised context — target encoding not applicable | Target encoding | ✅ |
| 8 | Save feature_cols.csv | Enables notebook 05/06 to reconstruct profiles without re-running this notebook | Hardcode column list | ✅ |

## 13. Next Steps — Lesson 4 Preview

- Load `X_scaled.npy` from `data/processed/`
- Apply PCA (2 components) → scree plot + 2D visualization
- Apply t-SNE on PCA-reduced data → visualize cluster structure
- Run KMeans elbow method (k=2…10)
- Apply KMeans, DBSCAN, and Hierarchical clustering
- Save cluster labels to `data/final/`

---

**← Previous:** [02 — Data Understanding](./02_data_understanding.ipynb)  
**Next →** [04 — Modeling](./04_modeling.ipynb)